In [7]:
import pandas as pd
import numpy as np
import os
import duckdb

In [12]:
os.getcwd()
con.close()

In [10]:
print(duckdb.__version__)

con = duckdb.connect("airflow-project/pizza_store.duckdb")

1.5.4


In [14]:
con.execute("""
CREATE TABLE pizzas_raw AS
SELECT *
FROM read_csv_auto(
    'data_sources/pizzas.csv'
);
""")

In [ ]:
con.execute("""
CREATE TABLE orders_raw AS
SELECT *
FROM read_csv_auto('data_sources/orders.csv');
""")

In [19]:
con.sql("""
ALTER TABLE orders_details_raw
RENAME TO order_details_raw;
""")

In [11]:
con.execute("SHOW TABLES").fetchall()

[('order_details_raw',),
 ('orders_raw',),
 ('pizza_types_raw',),
 ('pizzas_raw',)]

In [36]:
pizza_df = con.sql("""select * from pizza_types_raw""").df()

order_detail_df = con.sql("""select * from order_details_raw""").df()

In [47]:
display(con.sql("select * from stg_orders limit 1"))
display(con.sql("select * from stg_order_details limit 1"))
display(con.sql("select * from stg_pizzas limit 1"))
display(con.sql("select * from stg_pizza_types limit 1"))

┌──────────┬────────────┬────────────┐
│ order_id │ order_date │ order_time │
│  int64   │    date    │    time    │
├──────────┼────────────┼────────────┤
│        1 │ 2015-01-01 │ 11:38:36   │
└──────────┴────────────┴────────────┘

┌──────────────────┬──────────┬────────────┬──────────┐
│ order_details_id │ order_id │  pizza_id  │ quantity │
│      int64       │  int64   │  varchar   │  int64   │
├──────────────────┼──────────┼────────────┼──────────┤
│                1 │        1 │ hawaiian_m │        1 │
└──────────────────┴──────────┴────────────┴──────────┘

┌───────────┬───────────────┬────────────┬─────────────┐
│ pizza_id  │ pizza_type_id │ pizza_size │ pizza_price │
│  varchar  │    varchar    │  varchar   │    float    │
├───────────┼───────────────┼────────────┼─────────────┤
│ bbq_ckn_s │ bbq_ckn       │ S          │       12.75 │
└───────────┴───────────────┴────────────┴─────────────┘

┌───────────────┬────────────────────────────┬────────────────┬─────────────────────────────────────────────────────────────────────────────────────┐
│ pizza_type_id │         pizza_name         │ pizza_category │                                     ingredients                                     │
│    varchar    │          varchar           │    varchar     │                                       varchar                                       │
├───────────────┼────────────────────────────┼────────────────┼─────────────────────────────────────────────────────────────────────────────────────┤
│ bbq_ckn       │ The Barbecue Chicken Pizza │ Chicken        │ Barbecued Chicken, Red Peppers, Green Peppers, Tomatoes, Red Onions, Barbecue Sauce │
└───────────────┴────────────────────────────┴────────────────┴─────────────────────────────────────────────────────────────────────────────────────┘

In [92]:
#con.sql("select order_id, count(order_id) as num_of_order_id from stg_order_details group by 1 having count(order_id) > 1")

# pizza_type_id_df = con.sql("""
# select
    # pizza_type_id 
# from
    # stg_pizzas piz
# """).to_df()

pizza_type_id_table_df = con.sql("""
select
    pizza_type_id
from
    stg_pizza_types""").to_df()

In [4]:
# con = duckdb.connect("pizza_store.duckdb")
con.close()

In [125]:
con.sql("""
select 
    *
From daily_revenue
order by daily_revenue_amnt desc        
""")



┌────────────┬────────────────────┐
│ order_date │ daily_revenue_amnt │
│    date    │       float        │
├────────────┼────────────────────┤
│ 2015-11-27 │            4422.45 │
│ 2015-11-26 │            4405.95 │
│ 2015-10-15 │             4320.2 │
│ 2015-07-04 │             3864.2 │
│ 2015-07-03 │             3443.0 │
│ 2015-05-15 │            3386.15 │
│ 2015-07-24 │             3204.4 │
│ 2015-10-01 │            3202.15 │
│ 2015-02-01 │             3189.2 │
│ 2015-11-06 │             3157.5 │
│     ·      │                ·   │
│     ·      │                ·   │
│     ·      │                ·   │
│ 2015-02-22 │            1579.95 │
│ 2015-06-28 │             1569.7 │
│ 2015-04-19 │            1527.95 │
│ 2015-08-30 │             1494.6 │
│ 2015-09-06 │            1491.65 │
│ 2015-12-27 │             1419.0 │
│ 2015-11-22 │             1368.7 │
│ 2015-12-29 │            1353.25 │
│ 2015-12-30 │             1337.8 │
│ 2015-03-22 │            1259.25 │
└────────────┴──────────────

In [114]:
con.sql("""
select 
    order_date
    ,order_details_id
    ,order_id
    ,pizza_id
    ,quantity
    ,charge_by_order_details_id
from
    int_pizza_orders
where order_date = date('2015-01-01')
order by 
    order_date
""")

con.sql("""
select 
    order_date
    ,order_details_id
    ,order_id
    ,pizza_id
    ,charge_by_order_details_id
from 
    int_pizza_orders
where
     order_date = date('2015-01-01')

""")

┌────────────┬──────────────────┬──────────┬────────────────┬────────────────────────────┐
│ order_date │ order_details_id │ order_id │    pizza_id    │ charge_by_order_details_id │
│    date    │      int64       │  int64   │    varchar     │           float            │
├────────────┼──────────────────┼──────────┼────────────────┼────────────────────────────┤
│ 2015-01-01 │                1 │        1 │ hawaiian_m     │                      13.25 │
│ 2015-01-01 │                2 │        2 │ classic_dlx_m  │                       16.0 │
│ 2015-01-01 │                3 │        2 │ five_cheese_l  │                       18.5 │
│ 2015-01-01 │                4 │        2 │ ital_supr_l    │                      20.75 │
│ 2015-01-01 │                5 │        2 │ mexicana_m     │                       16.0 │
│ 2015-01-01 │                6 │        2 │ thai_ckn_l     │                      20.75 │
│ 2015-01-01 │                7 │        3 │ ital_supr_m    │                       16.5 │

In [27]:
# data_source_file_path = r"C:\Users\DELL\Desktop\Analytics-Engineering\data_sources"

# os.listdir(data_source_file_path)

In [ ]:
# dictionary = {}
# for i in os.listdir(data_source_file_path):
    # dictionary[i] = pd.read_csv(f'data_sources/{i}', encoding='latin1')

# dictionary

In [ ]:
# order_details_df = dictionary['order_details.csv']
# orders_df = dictionary['orders.csv']
# pizza_types_df = dictionary['pizza_types.csv']
# pizzas_df = dictionary['pizzas.csv']

In [15]:
# display(order_details_df)
# display(orders_df)
# display(pizza_types_df)
# display(pizzas_df)
